<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-34.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 34 - Flume y Sqoop: Pipeline de Ingestión

**Módulo:** Sistemas de Big Data (SBD - 5074)

---

## Objetivo

En este notebook simularemos un **pipeline completo de ingestión de datos**:

1. Crear una base de datos SQLite como fuente RDBMS (simulando MySQL/PostgreSQL)
2. Implementar operaciones tipo Sqoop (SQL → archivos)
3. Simular el flujo staging → validación → clean
4. Validar la integridad de los datos importados

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import hashlib
import time
import json
import shutil

np.random.seed(42)
print("Librerías cargadas correctamente ✅")

---

## 1. Crear la base de datos fuente (simula el RDBMS)

Creamos una base de datos SQLite con varias tablas, como tendríamos en MySQL o PostgreSQL en un entorno real.

In [ ]:
os.makedirs('pipeline', exist_ok=True)

conn = sqlite3.connect('pipeline/tienda.db')
cursor = conn.cursor()

cursor.executescript('''
    DROP TABLE IF EXISTS clientes;
    DROP TABLE IF EXISTS productos;
    DROP TABLE IF EXISTS ventas;

    CREATE TABLE clientes (
        id_cliente INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        email TEXT,
        ciudad TEXT,
        fecha_registro DATE
    );

    CREATE TABLE productos (
        id_producto INTEGER PRIMARY KEY,
        nombre TEXT NOT NULL,
        categoria TEXT,
        precio REAL
    );

    CREATE TABLE ventas (
        id_venta INTEGER PRIMARY KEY,
        id_cliente INTEGER,
        id_producto INTEGER,
        cantidad INTEGER,
        fecha DATE,
        total REAL,
        FOREIGN KEY (id_cliente) REFERENCES clientes(id_cliente),
        FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
    );
''')

N_CLIENTES = 2000
N_PRODUCTOS = 100
N_VENTAS = 50000

nombres = ['Ana', 'Carlos', 'María', 'Pedro', 'Laura', 'David', 'Elena', 'Miguel']
apellidos = ['García', 'López', 'Rodríguez', 'Martínez', 'Sánchez', 'Fernández']
ciudades = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao', 'Málaga']
categorias_prod = ['Electrónica', 'Ropa', 'Alimentación', 'Hogar', 'Deportes']

for i in range(1, N_CLIENTES + 1):
    nombre = f"{np.random.choice(nombres)} {np.random.choice(apellidos)}"
    email = f"cliente{i}@email.com"
    ciudad = np.random.choice(ciudades)
    fecha = f"2024-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}"
    cursor.execute('INSERT INTO clientes VALUES (?, ?, ?, ?, ?)',
                   (i, nombre, email, ciudad, fecha))

for i in range(1, N_PRODUCTOS + 1):
    nombre = f"Producto_{i:03d}"
    categoria = np.random.choice(categorias_prod)
    precio = round(np.random.uniform(5, 500), 2)
    cursor.execute('INSERT INTO productos VALUES (?, ?, ?, ?)',
                   (i, nombre, categoria, precio))

for i in range(1, N_VENTAS + 1):
    id_cliente = np.random.randint(1, N_CLIENTES + 1)
    id_producto = np.random.randint(1, N_PRODUCTOS + 1)
    cantidad = np.random.randint(1, 20)
    mes = np.random.randint(1, 13)
    dia = np.random.randint(1, 29)
    fecha = f"2025-{mes:02d}-{dia:02d}"
    precio_prod = cursor.execute('SELECT precio FROM productos WHERE id_producto=?',
                                 (id_producto,)).fetchone()[0]
    total = round(cantidad * precio_prod, 2)
    cursor.execute('INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?)',
                   (i, id_cliente, id_producto, cantidad, fecha, total))

conn.commit()

print("Base de datos creada ✅")
for tabla in ['clientes', 'productos', 'ventas']:
    count = cursor.execute(f'SELECT COUNT(*) FROM {tabla}').fetchone()[0]
    print(f"  Tabla '{tabla}': {count:,} registros")

---

## 2. Importación tipo Sqoop: SQL → Archivos

Simulamos el proceso de Sqoop: leer datos del RDBMS y escribirlos en archivos en el "HDFS" (sistema de archivos local como simulación).

In [ ]:
def sqoop_import(conn, tabla, target_dir, formato='parquet', where=None, num_mappers=1):
    """
    Simula una importación tipo Sqoop.
    Lee datos de SQLite y los escribe en archivos locales.
    """
    inicio = time.time()
    os.makedirs(target_dir, exist_ok=True)

    query = f"SELECT * FROM {tabla}"
    if where:
        query += f" WHERE {where}"

    df = pd.read_sql(query, conn)

    # Simular paralelismo con num_mappers (dividir en chunks)
    chunks = np.array_split(df, num_mappers)

    archivos_generados = []
    for i, chunk in enumerate(chunks):
        if formato == 'parquet':
            ruta = os.path.join(target_dir, f'part-{i:04d}.parquet')
            chunk.to_parquet(ruta, index=False)
        elif formato == 'csv':
            ruta = os.path.join(target_dir, f'part-{i:04d}.csv')
            chunk.to_csv(ruta, index=False)
        elif formato == 'json':
            ruta = os.path.join(target_dir, f'part-{i:04d}.json')
            chunk.to_json(ruta, orient='records', lines=True)
        archivos_generados.append(ruta)

    duracion = time.time() - inicio

    resultado = {
        'tabla': tabla,
        'registros': len(df),
        'formato': formato,
        'mappers': num_mappers,
        'archivos': archivos_generados,
        'duracion': duracion,
        'target_dir': target_dir
    }

    return resultado

# Importar las 3 tablas al directorio "raw" (staging)
for directorio in ['pipeline/raw', 'pipeline/staging', 'pipeline/clean']:
    if os.path.exists(directorio):
        shutil.rmtree(directorio)

print("=" * 60)
print("SQOOP IMPORT - Importando tablas al data lake")
print("=" * 60)

tablas_config = [
    {'tabla': 'clientes', 'formato': 'parquet', 'mappers': 2},
    {'tabla': 'productos', 'formato': 'parquet', 'mappers': 1},
    {'tabla': 'ventas', 'formato': 'parquet', 'mappers': 4},
]

resultados_import = {}
for config in tablas_config:
    target = f"pipeline/raw/{config['tabla']}"
    resultado = sqoop_import(conn, config['tabla'], target,
                             formato=config['formato'],
                             num_mappers=config['mappers'])
    resultados_import[config['tabla']] = resultado
    print(f"\n✅ Tabla '{config['tabla']}':")
    print(f"   Registros: {resultado['registros']:,}")
    print(f"   Formato: {resultado['formato']}")
    print(f"   Mappers: {resultado['mappers']} → {len(resultado['archivos'])} archivos")
    print(f"   Duración: {resultado['duracion']:.3f}s")

---

## 3. Validación post-importación

Verificamos que los datos importados son correctos y completos.

In [ ]:
def validar_importacion(conn, tabla, target_dir):
    """Ejecuta validaciones post-importación."""
    resultados = {'tabla': tabla, 'validaciones': []}

    # 1. Validación de conteo
    count_origen = pd.read_sql(f'SELECT COUNT(*) as n FROM {tabla}', conn)['n'][0]
    archivos = [os.path.join(target_dir, f) for f in os.listdir(target_dir) if f.endswith('.parquet')]
    df_destino = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
    count_destino = len(df_destino)

    conteo_ok = count_origen == count_destino
    resultados['validaciones'].append({
        'test': 'Conteo de registros',
        'origen': count_origen,
        'destino': count_destino,
        'resultado': '✅ PASS' if conteo_ok else '❌ FAIL'
    })

    # 2. Validación de schema
    cols_origen = set(pd.read_sql(f'SELECT * FROM {tabla} LIMIT 1', conn).columns)
    cols_destino = set(df_destino.columns)
    schema_ok = cols_origen == cols_destino
    resultados['validaciones'].append({
        'test': 'Schema (columnas)',
        'origen': len(cols_origen),
        'destino': len(cols_destino),
        'resultado': '✅ PASS' if schema_ok else f'❌ FAIL (diff: {cols_origen.symmetric_difference(cols_destino)})'
    })

    # 3. Validación de nulos inesperados
    nulos_destino = df_destino.isnull().sum().sum()
    df_origen = pd.read_sql(f'SELECT * FROM {tabla}', conn)
    nulos_origen = df_origen.isnull().sum().sum()
    nulos_ok = nulos_destino == nulos_origen
    resultados['validaciones'].append({
        'test': 'Nulos inesperados',
        'origen': int(nulos_origen),
        'destino': int(nulos_destino),
        'resultado': '✅ PASS' if nulos_ok else '❌ FAIL'
    })

    # 4. Checksum de la primera columna numérica
    for col in df_origen.select_dtypes(include=[np.number]).columns:
        sum_origen = df_origen[col].sum()
        sum_destino = df_destino[col].astype(float).sum()
        checksum_ok = abs(sum_origen - sum_destino) < 0.01
        resultados['validaciones'].append({
            'test': f'Checksum ({col})',
            'origen': round(sum_origen, 2),
            'destino': round(sum_destino, 2),
            'resultado': '✅ PASS' if checksum_ok else '❌ FAIL'
        })
        break

    return resultados

print("=" * 65)
print("VALIDACIÓN POST-IMPORTACIÓN")
print("=" * 65)

for tabla in ['clientes', 'productos', 'ventas']:
    target_dir = f'pipeline/raw/{tabla}'
    resultado = validar_importacion(conn, tabla, target_dir)

    print(f"\n--- Tabla: {tabla} ---")
    for v in resultado['validaciones']:
        print(f"  {v['resultado']} {v['test']}: origen={v['origen']}, destino={v['destino']}")

---

## 4. Pipeline staging → clean

Simulamos la fase de limpieza: tomamos los datos "raw", aplicamos transformaciones y guardamos en "clean".

In [ ]:
def cargar_tabla_raw(tabla):
    """Carga todos los partfiles de una tabla."""
    target_dir = f'pipeline/raw/{tabla}'
    archivos = sorted([os.path.join(target_dir, f) for f in os.listdir(target_dir) if f.endswith('.parquet')])
    return pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)

df_ventas = cargar_tabla_raw('ventas')
df_clientes = cargar_tabla_raw('clientes')
df_productos = cargar_tabla_raw('productos')

print("=" * 60)
print("PIPELINE: RAW → STAGING → CLEAN")
print("=" * 60)

# Fase 1: Join (enriquecer ventas con datos de cliente y producto)
df_enriquecido = df_ventas.merge(
    df_clientes[['id_cliente', 'nombre', 'ciudad']],
    on='id_cliente', how='left', suffixes=('', '_cliente')
).merge(
    df_productos[['id_producto', 'nombre', 'categoria', 'precio']],
    on='id_producto', how='left', suffixes=('_cliente', '_producto')
)

print(f"\n1️⃣ Enriquecimiento (JOIN): {len(df_ventas):,} ventas → {len(df_enriquecido):,} registros enriquecidos")
print(f"   Columnas: {list(df_enriquecido.columns)}")

# Fase 2: Limpieza básica
antes = len(df_enriquecido)
df_enriquecido = df_enriquecido.drop_duplicates()
df_enriquecido = df_enriquecido.dropna(subset=['id_cliente', 'id_producto', 'total'])
df_enriquecido = df_enriquecido[df_enriquecido['total'] > 0]
despues = len(df_enriquecido)
print(f"\n2️⃣ Limpieza: {antes:,} → {despues:,} registros ({antes-despues} eliminados)")

# Fase 3: Guardar en directorio clean con particionamiento por mes
df_enriquecido['mes'] = pd.to_datetime(df_enriquecido['fecha']).dt.to_period('M').astype(str)

os.makedirs('pipeline/clean/ventas_enriquecidas', exist_ok=True)
for mes, grupo in df_enriquecido.groupby('mes'):
    ruta = f'pipeline/clean/ventas_enriquecidas/mes={mes}'
    os.makedirs(ruta, exist_ok=True)
    grupo.drop(columns=['mes']).to_parquet(f'{ruta}/datos.parquet', index=False)

particiones = os.listdir('pipeline/clean/ventas_enriquecidas')
print(f"\n3️⃣ Guardado particionado: {len(particiones)} particiones por mes")
for p in sorted(particiones)[:5]:
    print(f"   📁 {p}")
if len(particiones) > 5:
    print(f"   ... y {len(particiones) - 5} más")

print("\n✅ Pipeline completado")

---

## 5. Importación incremental

Simulamos una importación incremental tipo Sqoop: solo traemos registros **nuevos** desde la última importación.

In [ ]:
# Añadir nuevas ventas al RDBMS (simulando actividad diaria)
nuevas_ventas = []
for i in range(N_VENTAS + 1, N_VENTAS + 501):
    id_cliente = np.random.randint(1, N_CLIENTES + 1)
    id_producto = np.random.randint(1, N_PRODUCTOS + 1)
    cantidad = np.random.randint(1, 20)
    precio_prod = cursor.execute('SELECT precio FROM productos WHERE id_producto=?',
                                  (id_producto,)).fetchone()[0]
    total = round(cantidad * precio_prod, 2)
    cursor.execute('INSERT INTO ventas VALUES (?, ?, ?, ?, ?, ?)',
                   (i, id_cliente, id_producto, cantidad, '2026-02-16', total))

conn.commit()
print(f"Se añadieron 500 nuevas ventas al RDBMS")

# Importación incremental: solo registros con id > último importado
ultimo_id_importado = N_VENTAS

resultado_incr = sqoop_import(
    conn, 'ventas',
    'pipeline/raw/ventas_incremental',
    formato='parquet',
    where=f'id_venta > {ultimo_id_importado}',
    num_mappers=1
)

print(f"\n=== IMPORTACIÓN INCREMENTAL ===")
print(f"Último ID importado: {ultimo_id_importado:,}")
print(f"Nuevos registros importados: {resultado_incr['registros']:,}")
print(f"Duración: {resultado_incr['duracion']:.3f}s")

total_rdbms = cursor.execute('SELECT COUNT(*) FROM ventas').fetchone()[0]
print(f"\nTotal en RDBMS: {total_rdbms:,}")
print(f"Total importado (full + incremental): {N_VENTAS + resultado_incr['registros']:,}")

---

## 6. Resumen del pipeline

Visualizamos las estadísticas del pipeline completo.

In [ ]:
import matplotlib.pyplot as plt

def calcular_tamano_directorio(directorio):
    total = 0
    for dirpath, _, filenames in os.walk(directorio):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / 1024  # KB

etapas = ['RDBMS\n(SQLite)', 'Raw\n(Parquet)', 'Clean\n(Particionado)']
registros = [
    cursor.execute('SELECT COUNT(*) FROM ventas').fetchone()[0],
    N_VENTAS,
    len(df_enriquecido)
]

tamano_db = os.path.getsize('pipeline/tienda.db') / 1024
tamano_raw = calcular_tamano_directorio('pipeline/raw')
tamano_clean = calcular_tamano_directorio('pipeline/clean')
tamanos = [tamano_db, tamano_raw, tamano_clean]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colores = ['#3498db', '#e67e22', '#2ecc71']

bars1 = ax1.bar(etapas, registros, color=colores, edgecolor='black', linewidth=0.5)
ax1.set_title('Registros por etapa del pipeline', fontweight='bold')
ax1.set_ylabel('Número de registros')
for bar, val in zip(bars1, registros):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
             f'{val:,}', ha='center', fontweight='bold')

bars2 = ax2.bar(etapas, tamanos, color=colores, edgecolor='black', linewidth=0.5)
ax2.set_title('Tamaño por etapa (KB)', fontweight='bold')
ax2.set_ylabel('Tamaño (KB)')
for bar, val in zip(bars2, tamanos):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val:.0f}', ha='center', fontweight='bold')

plt.suptitle('Resumen del Pipeline de Ingestión', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 Resumen:")
print(f"   RDBMS → Raw: importación completa de {N_VENTAS:,} registros")
print(f"   Raw → Clean: limpieza + enriquecimiento + particionamiento")
print(f"   Compresión Parquet: {tamano_db/tamano_raw:.1f}x vs SQLite original")

conn.close()

---

## 7. Resumen y conclusiones

| Herramienta | Tipo | Dirección | Caso de uso |
|-------------|------|-----------|-------------|
| **Sqoop** | Batch | RDBMS ↔ HDFS | Importar/exportar tablas completas |
| **Flume** | Streaming | Sources → HDFS | Ingestión continua de logs/eventos |

### Pipeline estándar:
1. **Extraer** (Sqoop/Flume) → archivos en `/raw/`
2. **Validar** (conteos, checksums, schema)
3. **Limpiar** (nulos, duplicados, enriquecimiento)
4. **Almacenar** → archivos Parquet particionados en `/clean/`

---

## Ejercicios para practicar

1. Añade una validación de checksum MD5 sobre los archivos Parquet generados.
2. Implementa un log de auditoría que registre cada importación (tabla, fecha, registros, duración).
3. Simula un fallo a mitad de la importación y diseña un mecanismo de recuperación.